<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/DecisionIntelligenceWorkshopLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Decisions with Reusable Prompt Functions</span>
</div>

Decision Intelligence applied in this module:  
* Listing various decision-making frameworks with their descriptions 
* Listing types of decision-making frameworks dynamically based on the decision type
* Reusing AI Extensions prompt templates with dynamic inputs

AI Extensions prompt templates are reusable instructions that help guide GenAI models toward a specific decision workflow. A prompt template usually has one responsibility: list useful decision frameworks, recommend a reasoning approach for a high-stakes decision, summarize a tradeoff, or personalize a recommendation from structured context.

What makes reusable prompt functions useful? GenAI models can already respond to basic prompts, but decision workflows usually need repeatable instructions, consistent formatting, and dynamic inputs. By combining Microsoft.Extensions.AI with simple C# helper methods, a prompt becomes a reusable part of an AI orchestration flow instead of a one-off string.

For example, imagine you want to prepare a great Thanksgiving dinner and you ask a GenAI cooking application to create a recipe. It may produce a delicious plan, but it may also use ingredients or cooking appliances you do not own. You could keep prompting until it narrows the recipe, but it is better to provide available ingredients, time availability, kitchen appliances, and allergy preferences as dynamic inputs. The same pattern applies to Decision Intelligence: reusable prompt templates let the model consider specific business context, user constraints, and decision criteria each time the prompt runs.

In this notebook, reusable prompt functions are implemented with:
* `IChatClient` from Microsoft.Extensions.AI
* `ChatMessage` roles for system and user instructions
* `ChatOptions` for model settings such as temperature, top-p, and maximum output tokens 
* C# raw string interpolation for dynamic prompt inputs

Below is an example of a reusable prompt template that can be used for facilitating decision-making. Note the `{decisionToMake}` input where the parameter can be dynamically passed in to provide additional specificity.

```csharp
var decisionPrompt = $"""
[DECISION FRAMEWORKS TO USE]
Price's Law
Pareto Principle
Laplace Rule of Succession
Eisenhower Matrix
Median Rule of Five
Second Order Thinking

[DECISION GUIDANCE]
Try to use the best fitting framework.  
Prefer using quantitative decision frameworks rather than qualitative ones.  
Use calculations or code-based validation when quantitative reasoning is required.

Use the decision frameworks above to help the user make the following decision:
{decisionToMake}
""";
```

The core value is that the prompt can be reused with different inputs while still flowing through the same AI Extensions `IChatClient` API.

----
### Step 1 - Initialize Configuration Builder & Build the AI Extensions Responses API Orchestration 

Execute the next two cells to:
* Use the Configuration Builder to load the API secrets.  
* Use the API configuration to build a Responses API-backed AI Extensions `IChatClient` orchestration.

In [1]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.8"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.8"
#r "nuget: System.Text.Json, 10.0.8"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.8 Microsoft.Extensions.Configuration.Json, 10.0.8 System.Text.Json, 10.0.8

In [2]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.6.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Responses;
using System.ClientModel;
using System.Collections.Generic;
using System.Threading.Tasks;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

#pragma warning disable OPENAI001

if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    // Get the ResponsesClient from AzureOpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(openAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    // Get the ResponsesClient from OpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

var decisionIntelligenceSystemPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems,
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
""";

async Task<string> RunPromptAsync(string prompt, ChatOptions? options = null)
{
    ChatResponse response = await chatClient.GetResponseAsync(prompt, options);
    return response.Text ?? string.Empty;
}

async Task<string> RunDecisionPromptAsync(string userPrompt, ChatOptions? options = null)
{
    var chatMessages = new List<ChatMessage>
    {
        new(ChatRole.System, decisionIntelligenceSystemPrompt),
        new(ChatRole.User, userPrompt)
    };

    ChatResponse response = await chatClient.GetResponseAsync(chatMessages, options);
    return response.Text ?? string.Empty;
}

Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.6.0 Microsoft.Extensions.AI.Abstractions, 10.6.0 Microsoft.Extensions.AI.OpenAI, 10.6.0 OpenAI, 2.10.0

Using Azure OpenAI Service



(57,61): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(63,73): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Azure.Core, Version=1.51.1.0, Culture=neutral, PublicK

----
### Step 2 - Create an AI Extensions Reusable Prompt Function 

> 📜 **_"Decision frameworks are like maps. Use them to navigate complex decision-making terrain._** 
>
> -- <cite>Unknown</cite> 

Reusable prompt functions in this notebook are simple C# helper methods that submit prompt templates through `IChatClient`. This keeps the prompt reusable while using the same AI Extensions API surface shown in the previous workshop notebooks.

In [3]:
// Simple prompt to list some decision frameworks this GenAI LLM is familiar with
var decisionFrameworksPromptTemplate = """
Identify and list various decision-making frameworks that can enhance the quality of decisions. 
Briefly describe how each framework supports better analysis and reasoning in different decision-making scenarios.

Output Format Instructions: 
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.
Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Treat the prompt template as a reusable local function that invokes the AI Extensions IChatClient.
async Task<string> RunDecisionFrameworksPromptAsync(ChatOptions? options = null)
{
    return await RunDecisionPromptAsync(decisionFrameworksPromptTemplate, options);
}

var decisionFrameworksResponseText = await RunDecisionFrameworksPromptAsync();
decisionFrameworksResponseText.DisplayAs("text/markdown");

| Framework | What it is | How it improves decision quality | Best used when |
|---|---|---|---|
| SWOT Analysis | A structured review of Strengths, Weaknesses, Opportunities, and Threats. | Helps organize internal and external factors so decisions consider capabilities and risks together. | Strategic planning, market entry, product launches. |
| Cost-Benefit Analysis | Compares expected benefits and costs, often in monetary terms. | Makes tradeoffs explicit and helps determine whether a choice creates net value. | Budgeting, investments, operational changes. |
| Decision Matrix / Weighted Scoring | Scores options against criteria and weights them by importance. | Reduces bias by using a repeatable, criteria-based comparison across alternatives. | Comparing vendors, hires, projects, or products. |
| OODA Loop | Observe, Orient, Decide, Act in a continuous cycle. | Supports fast learning and adaptation in dynamic environments by combining action with feedback. | Competitive, fast-changing, or uncertain situations. |
| Pareto Analysis | Focuses on the small number of causes that produce most effects, often the 80/20 rule. | Helps prioritize the highest-impact issues instead of spreading attention too thin. | Problem solving, quality improvement, resource allocation. |
| Root Cause Analysis | Identifies the underlying cause of a problem rather than treating symptoms. | Prevents shallow fixes by improving diagnostic accuracy and long-term effectiveness. | Recurring operational issues, failures, defects. |
| Pre-mortem Analysis | Assumes a decision failed and asks why. | Surfaces hidden risks, blind spots, and failure modes before commitment. | High-stakes decisions, launches, major investments. |
| Scenario Planning | Builds multiple plausible future scenarios and tests decisions against them. | Strengthens reasoning under uncertainty by exploring how choices perform in different futures. | Long-term strategy, policy, supply chain, forecasting. |
| Second-Order Thinking | Examines not just immediate effects, but downstream consequences. | Helps avoid short-sighted choices by considering ripple effects and unintended outcomes. | Policy, pricing, incentives, systems with feedback loops. |
| Pros and Cons List | Enumerates advantages and disadvantages of each option. | Provides a simple starting structure that clarifies arguments and exposes major tradeoffs. | Quick personal decisions, early-stage comparison. |
| STEEP/PESTLE Analysis | Reviews Social, Technological, Economic, Environmental, Political, and Legal factors. | Expands analysis beyond the immediate problem to external forces that may shape outcomes. | Macro-level strategy, market analysis, regulation-sensitive decisions. |
| RICE Scoring | Prioritization method using Reach, Impact, Confidence, and Effort. | Balances value, certainty, and cost, making prioritization more disciplined. | Product roadmaps, feature prioritization, initiatives. |
| Expected Value Analysis | Weighs possible outcomes by probability and consequence. | Improves decisions under uncertainty by quantifying risk-adjusted outcomes. | Investments, games of chance, uncertain operational choices. |
| Bayesian Thinking | Updates beliefs as new evidence arrives. | Encourages probabilistic reasoning and continuous refinement of judgment. | Forecasting, diagnosis, ambiguous evidence, learning systems. |
| Devil’s Advocate / Red Teaming | Intentionally challenges assumptions and preferred options. | Reduces groupthink and overconfidence by stress-testing reasoning from the opposite side. | Security, strategy, governance, high-risk decisions. |
| Multi-Criteria Decision Analysis (MCDA) | Evaluates options across several dimensions, not just one. | Supports balanced decision-making when goals conflict or are difficult to quantify. | Complex tradeoff decisions with many stakeholders. |
| Cynefin Framework | Classifies situations as simple, complicated, complex, chaotic, or disorderly. | Helps match the decision approach to the type of problem, reducing misuse of rigid methods. | Ambiguous, uncertain, or rapidly changing contexts. |
| DMAIC | Define, Measure, Analyze, Improve, Control. | Provides a disciplined improvement process that turns vague problems into measurable interventions. | Process improvement, operations, quality management. |
| Heuristics with Guardrails | Simple rules of thumb combined with checks to limit error. | Speeds decisions while preserving quality through constraints and review points. | Repetitive decisions, time-sensitive operations. |


(17,64): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



----
### Step 3 - AI Extensions Dynamic Decision Intelligence  

> 📜 **_"If the only tool you have is a hammer, you tend to see every problem as a nail."_**
>
> -- <cite>Abraham Maslow (Renowned American psychologist)</cite> 

Prompt templates can be dynamically composed using C# variables and raw string interpolation. This allows the ease of passing in parameters and execution settings into reusable prompt functions. This is not groundbreaking, but it does allow you to dynamically compose AI prompts (context-engineering, prompt-engineering dynamically).

Execute the cell below to view how the prompt can dynamically instruct the LLM to retrieve different types of decision-making frameworks.

In [ ]:
// Takes input variables and creates a dynamic prompt that can be used to invoke the GenAI model.
string CreateDecisionFrameworkPrompt(int numberOfFrameworks, string decisionType)
{
    return $"""
        Identify and list {numberOfFrameworks} decision-making frameworks that can enhance the quality of decisions. 
        Briefly describe how each framework supports better analysis and reasoning in {decisionType} decision-making scenarios.

        Output Format Instructions: 
        When generating Markdown, do not use any headings higher than ###. 
        Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
        All top-level section headers should start at ### or lower. 
        Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
        For separation, use extra extra spacing. Do not any render horizontal lines.
        Format the response using only a Markdown table. Only return a Markdown table. 
        Do not enclose the table in triple backticks.
        """;
}

// Create the chat options. 
//Try changing the settings to see how they affect the quality of generated text.
#pragma warning disable OPENAI001

var decisionFrameworkChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

#pragma warning restore OPENAI001

// Dynamically set the number of frameworks and decision type -> strategic
var strategicDecisionFrameworkPrompt = CreateDecisionFrameworkPrompt(
    numberOfFrameworks: 3,
    decisionType: "Long-Term strategic with probabilistic outcomes");

var strategicDecisionFrameworkResponseText = await RunDecisionPromptAsync(
    strategicDecisionFrameworkPrompt,
    decisionFrameworkChatOptions);

strategicDecisionFrameworkResponseText.DisplayAs("text/markdown");

| Framework | Core idea | How it supports better long-term strategic decisions with probabilistic outcomes |
|---|---|---|
| Expected Value / Expected Utility Analysis | Compare options by weighting possible outcomes by their probabilities, optionally adjusting for risk preferences. | Forces explicit thinking about uncertainty instead of relying on intuition. Helps rank long-term options by their likely overall value, making tradeoffs clearer when outcomes vary in probability and magnitude. |
| Scenario Planning | Develop multiple plausible futures and test how each strategy performs across them. | Improves reasoning under uncertainty by reducing dependence on a single forecast. Reveals robust strategies, key assumptions, and vulnerabilities, which is especially useful when long-term outcomes are highly uncertain and path-dependent. |
| Decision Trees / Bayesian Decision Analysis | Map choices, chance events, and outcomes in a branching structure; update probabilities as new evidence arrives. | Makes complex strategic decisions more transparent by structuring sequences of choices and uncertainties. Supports disciplined analysis of contingent actions, learning over time, and evidence-based updates to improve decisions as probabilities change. |

In the example below, the number of frameworks to return has been changed and the type has been changed to **Quick to Implement for rapid Decision-Making**.

In [6]:
// Dynamically set the number of frameworks and decision type -> requiring fast action
var rapidDecisionFrameworkPrompt = CreateDecisionFrameworkPrompt(
    numberOfFrameworks: 5,
    decisionType: "Quick to Implement for rapid Decision-Making");

// Note: The invocation has NOT changed, only the dynamic prompt inputs have changed.
var rapidDecisionFrameworkResponseText = await RunDecisionPromptAsync(
    rapidDecisionFrameworkPrompt,
    decisionFrameworkChatOptions);

rapidDecisionFrameworkResponseText.DisplayAs("text/markdown");

| Framework | Brief description | How it supports better analysis and reasoning in quick, rapid decision-making scenarios |
|---|---|---|
| OODA Loop (Observe–Orient–Decide–Act) | A fast-cycle framework for continuously updating your view of a situation and responding quickly. | Helps you make decisions under time pressure by forcing a rapid scan of the situation, a quick sense-making step, and immediate action with feedback. Useful when conditions are changing and waiting too long is costly. |
| Cost-Benefit Analysis | Compares expected gains and losses of each option, including time, money, risk, and effort. | Improves reasoning by making tradeoffs explicit and easy to compare. In fast decisions, it gives a simple structure for choosing the option with the best expected payoff relative to cost. |
| Eisenhower Matrix | Sorts actions by urgency and importance into four quadrants. | Supports quick prioritization by separating what needs immediate attention from what can wait or be delegated. It reduces overload and helps focus on decisions with the highest impact. |
| RAPID Decision Model | Clarifies decision roles: Recommend, Agree, Perform, Input, Decide. | Improves decision quality by defining who contributes information and who makes the final call. In fast-moving situations, it reduces confusion, speeds alignment, and avoids delays caused by unclear ownership. |
| SWOT Analysis | Evaluates Strengths, Weaknesses, Opportunities, and Threats of an option or situation. | Helps decision-makers quickly organize internal and external factors into a structured view. Even in short timeframes, it improves reasoning by revealing advantages, vulnerabilities, and risks before committing. |

----
### Step 4 - Decision Scenario with Dynamic Decision Intelligence

In the below code a decision-analysis scenario is introduced that uses dynamic inputs to personalize the decision recommendation dynamically. The more specific information that provides contextual information to the Generative AI model can greatly improve the specific recommendations. 

**Decision Scenario:** Michael is a 35-year-old professional chef who is considering opening his own restaurant. This is a significant life decision that could greatly impact his career and personal life. Michael is looking for a recommendation for an approach (decision) for this potentially life-changing decision. 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-RestaurantDecision.png">

Three factors will be considered for this decision scenario that the user will be prompted for: 
1) Michael's Total Net Worth in dollars (enter number)
2) Level of competition with other restaurants (low, medium, high) 
3) Level of support from Michael's friends and family (low, medium, high) 

**Various Decision Inputs:** You can simulate different What-If scenarios by varying the input of net worth, restaurant competition and level of family support. Note the different decision recommendations based on this provided by Artificial Intelligence. The recommendations in this example are quite simple, but even on the extreme inputs Generative AI has a concept of understanding the quality of inputs.

In [10]:
// Import the Microsoft.DotNet.Interactive namespace for user input
using Microsoft.DotNet.Interactive;

var totalNetWorth = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Michael's total net worth in dollars: ");
var levelOfCompetition = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Level of competition with other restaurants (Low, Medium, High): ");
var levelOfFamilySupport = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Level of support from Michael's friends and family (Low, Medium, High): ");

Console.WriteLine($"Michael's Net Worth: {totalNetWorth}");
Console.WriteLine($"Level of Restaurant Competition: {levelOfCompetition}");
Console.WriteLine($"Michael's level of Family Support: {levelOfFamilySupport}");
Console.WriteLine();

// Takes dynamic inputs and creates a personalized prompt that can be used to invoke the model.
var restaurantDecisionRecommendationPrompt = $"""
Michael is a 35-year-old professional chef who is considering opening his own restaurant. 
This is a significant life decision that could greatly impact his career and personal life. 
Michael is looking for a recommendation on how to approach this.
Some key information about Michael:
- Michael's total net worth is (in US dollars) ${totalNetWorth}.
- The level of competition with other restaurants is {levelOfCompetition}.
- The level of support from Michael's friends and family is {levelOfFamilySupport}.

Based on this information, what recommendation would you give to Michael regarding opening his own restaurant?

Output Format Instructions: 
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.
Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
Provide a small Decision Summary of the recommendation below the table.
""";


#pragma warning disable OPENAI001

// Create the chat options. Try changing the settings to see how they affect the quality of generated text.
var restaurantDecisionChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

var restaurantDecisionRecommendationResponseText = await RunDecisionPromptAsync(
    restaurantDecisionRecommendationPrompt,
    restaurantDecisionChatOptions);

restaurantDecisionRecommendationResponseText.DisplayAs("text/markdown");

Michael's Net Worth: 100000000000
Level of Restaurant Competition: High
Michael's level of Family Support: Low



| Factor | Assessment | Implication for Michael |
|---|---|---|
| Financial capacity | Extremely strong net worth ($100,000,000,000) | He can absorb startup losses, hire top talent, and fund a premium launch without personal financial strain. |
| Industry competition | High | The restaurant market is crowded, so success will depend on a distinct concept, excellent execution, and strong positioning. |
| Personal support | Low support from friends/family | A restaurant is demanding and stressful; low support increases emotional and operational risk, especially during the early stages. |
| Career fit | High, given he is already a professional chef | He likely has relevant expertise and credibility, which improves his odds compared with a non-industry entrant. |
| Recommended approach | Proceed, but cautiously and in stages | He should not rush into a full-scale standalone restaurant. A better path is to validate the concept first through a pop-up, chef’s table, residency, ghost kitchen, or small pilot location before committing to a major opening. |
| Decision Summary | Recommendation | Michael can afford to take the risk, but the combination of high competition and low personal support argues for a measured, test-first strategy rather than an immediate full-scale restaurant launch. |